# Taller 1 – Punto 3: Filtro Colaborativo Usuario-Usuario


## 0. Instalación e imports

In [3]:
# Instalar Surprise si no está disponible
# !pip install scikit-surprise

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, KNNWithZScore, accuracy
from surprise.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_absolute_error

print('Librerías cargadas correctamente ✓')

Librerías cargadas correctamente ✓


## 1. Carga y preprocesamiento de datos

In [13]:


## 1. Cargar dataset preprocesado

ratings = pd.read_csv("rating_sample.csv")[['userId','movieId','rating']]

print(f'Dataset preprocesado : {len(ratings):,} ratings')
print(f'Usuarios únicos      : {ratings.userId.nunique():,}')
print(f'Películas únicas     : {ratings.movieId.nunique():,}')

Dataset preprocesado : 1,836,903 ratings
Usuarios únicos      : 2,000
Películas únicas     : 11,886


In [14]:
# ── Cargar en formato Surprise ─────────────────────────────────────────────
reader = Reader(rating_scale=(0.5, 5.0))
surprise_data = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader)

# Split 80/20
trainset, testset = train_test_split(surprise_data, test_size=0.2, random_state=42)

print(f'Train: {trainset.n_ratings:,} ratings')
print(f'Test : {len(testset):,} ratings')

Train: 1,469,522 ratings
Test : 367,381 ratings


## 2. Punto 3b – Construcción de modelos usuario-usuario

Para calcular el rating se puede utilizar:
- **KNNBasic** – predicción ponderada simple
- **KNNWithMeans** – predicción centrada en la media del usuario (equivale a Pearson centrado)
- **KNNWithZScore** – predicción normalizada por desviación estándar

En este caso se utilizo la predicción centrada en la media del usuario, es decir, la predicción se calcula corrigiendo por la media del usuario.



La similitud Jaccard se calculó a partir de una matriz binaria usuario-ítem que indica si un usuario calificó o no una película. Posteriormente se calcularon las intersecciones y uniones entre usuarios para obtener la similitud Jaccard. Finalmente, las predicciones se realizaron utilizando un promedio ponderado de los ratings de los K vecinos más similares.

In [15]:
# ── Punto 3b-i: Jaccard ─────────────────────────────────────────────────────
# Surprise no incluye Jaccard directamente.

def jaccard_predict(test_df, train_df, k=10):
    """Predicción usuario-usuario con similitud Jaccard (implementación manual)."""
    # Matriz binaria de co-ocurrencia
    train_matrix = train_df.pivot(index='userId', columns='movieId', values='rating')
    binary = train_matrix.notna().astype(np.float32).values
    inter  = binary @ binary.T
    sums   = binary.sum(axis=1)
    union  = sums[:, None] + sums[None, :] - inter
    union[union == 0] = 1e-10
    jac_sim = pd.DataFrame(inter / union, index=train_matrix.index, columns=train_matrix.index)
    np.fill_diagonal(jac_sim.values, 0)

    user_means = train_matrix.mean(axis=1)
    preds, actuals = [], []

    for _, row in test_df.iterrows():
        u, m, r = row['userId'], row['movieId'], row['rating']
        if u not in jac_sim.index or m not in train_matrix.columns:
            continue
        neighbors = jac_sim.loc[u].nlargest(k)
        ratings_n = train_matrix.loc[neighbors.index, m].dropna()
        if ratings_n.empty:
            continue
        w = neighbors[ratings_n.index].values
        denom = np.abs(w).sum()
        if denom == 0:
            continue
        pred = np.clip(np.dot(w, ratings_n.values) / denom, 0.5, 5.0)
        preds.append(pred)
        actuals.append(r)

    mae  = mean_absolute_error(actuals, preds) if preds else np.nan
    rmse = np.sqrt(np.mean((np.array(actuals) - np.array(preds))**2)) if preds else np.nan
    cov  = len(preds) / len(test_df)
    return mae, rmse, cov

# Reconstruir train/test como DataFrames para Jaccard
inner2raw_user  = {trainset.to_raw_uid(i): i for i in trainset.all_users()}
train_rows = [(trainset.to_raw_uid(u), trainset.to_raw_iid(i), r)
              for u, i, r in trainset.all_ratings()]
train_df = pd.DataFrame(train_rows, columns=['userId','movieId','rating'])
test_df  = pd.DataFrame([(u, i, r) for u, i, r in testset], columns=['userId','movieId','rating'])

print('Calculando Jaccard...')
mae_jac, rmse_jac, cov_jac = jaccard_predict(test_df, train_df, k=10)
print(f'Jaccard  →  MAE={mae_jac:.4f}  RMSE={rmse_jac:.4f}  Cobertura={cov_jac:.1%}')

Calculando Jaccard...
Jaccard  →  MAE=0.7624  RMSE=0.9897  Cobertura=87.7%


In [16]:
# ── Punto 3b-ii: Coseno (Surprise) ─────────────────────────────────────────
sim_cosine = {'name': 'cosine', 'user_based': True}
algo_cos   = KNNWithMeans(k=10, min_k=2, sim_options=sim_cosine, verbose=False)
algo_cos.fit(trainset)
preds_cos  = algo_cos.test(testset)

mae_cos  = accuracy.mae(preds_cos,  verbose=False)
rmse_cos = accuracy.rmse(preds_cos, verbose=False)
cov_cos  = sum(1 for p in preds_cos if not p.details['was_impossible']) / len(preds_cos)

print(f'Coseno   →  MAE={mae_cos:.4f}  RMSE={rmse_cos:.4f}  Cobertura={cov_cos:.1%}')

Coseno   →  MAE=0.6317  RMSE=0.8152  Cobertura=100.0%


In [17]:
# ── Punto 3b-iii: Pearson (Surprise) ───────────────────────────────────────
sim_pearson = {'name': 'pearson', 'user_based': True}
algo_pea    = KNNWithMeans(k=10, min_k=2, sim_options=sim_pearson, verbose=False)
algo_pea.fit(trainset)
preds_pea   = algo_pea.test(testset)

mae_pea  = accuracy.mae(preds_pea,  verbose=False)
rmse_pea = accuracy.rmse(preds_pea, verbose=False)
cov_pea  = sum(1 for p in preds_pea if not p.details['was_impossible']) / len(preds_pea)

print(f'Pearson  →  MAE={mae_pea:.4f}  RMSE={rmse_pea:.4f}  Cobertura={cov_pea:.1%}')

Pearson  →  MAE=0.6268  RMSE=0.8171  Cobertura=100.0%


## 3. Punto 3c – Tabla comparativa de métricas

In [18]:
df_metrics = pd.DataFrame([
    {'Similitud': 'Jaccard',  'MAE': mae_jac,  'RMSE': rmse_jac,  'Cobertura': f'{cov_jac:.1%}'},
    {'Similitud': 'Coseno',   'MAE': mae_cos,  'RMSE': rmse_cos,  'Cobertura': f'{cov_cos:.1%}'},
    {'Similitud': 'Pearson',  'MAE': mae_pea,  'RMSE': rmse_pea,  'Cobertura': f'{cov_pea:.1%}'},
]).set_index('Similitud').round(4)

print('Comparación de métricas de similitud (k=10):')
df_metrics

Comparación de métricas de similitud (k=10):


,MAE,RMSE,Cobertura
Similitud,,,
Jaccard,0.7624,0.9897,87.7%
Coseno,0.6317,0.8152,100.0%
Pearson,0.6268,0.8171,100.0%


## 4. Punto 3d – Variación del número de vecinos K

In [19]:
k_values = [5, 10, 20, 50, 100]
rows_k = []

print(f"{'K':>5}  {'MAE_Cos':>10}  {'MAE_Pea':>10}  {'RMSE_Cos':>10}  {'RMSE_Pea':>10}")
print('-' * 52)

for k in k_values:
    # Coseno
    a = KNNWithMeans(k=k, min_k=2, sim_options={'name':'cosine','user_based':True}, verbose=False)
    a.fit(trainset)
    p = a.test(testset)
    mc = accuracy.mae(p, verbose=False)
    rc = accuracy.rmse(p, verbose=False)

    # Pearson
    b = KNNWithMeans(k=k, min_k=2, sim_options={'name':'pearson','user_based':True}, verbose=False)
    b.fit(trainset)
    q = b.test(testset)
    mp = accuracy.mae(q, verbose=False)
    rp = accuracy.rmse(q, verbose=False)

    rows_k.append({'K': k, 'MAE_Coseno': mc, 'MAE_Pearson': mp, 'RMSE_Coseno': rc, 'RMSE_Pearson': rp})
    print(f'{k:>5}  {mc:>10.4f}  {mp:>10.4f}  {rc:>10.4f}  {rp:>10.4f}')

df_k = pd.DataFrame(rows_k).set_index('K')
df_k

    K     MAE_Cos     MAE_Pea    RMSE_Cos    RMSE_Pea
----------------------------------------------------
    5      0.6470      0.6486      0.8338      0.8438
   10      0.6317      0.6268      0.8152      0.8171
   20      0.6236      0.6160      0.8064      0.8043
   50      0.6202      0.6116      0.8036      0.7993
  100      0.6214      0.6122      0.8061      0.7999


,MAE_Coseno,MAE_Pearson,RMSE_Coseno,RMSE_Pearson
K,,,,
5,0.646994,0.648608,0.833755,0.843758
10,0.631653,0.626808,0.815219,0.817098
20,0.623588,0.616020,0.806380,0.804314
50,0.620157,0.611623,0.803626,0.799274
100,0.621422,0.612166,0.806122,0.799856


## 5. Punto 3d – Variación por umbral de similitud (min_k)

In [20]:
# min_k es el mínimo de vecinos con rating para considerar la predicción válida.
# Aumentar min_k = umbral más estricto (mayor calidad, menor cobertura)

mink_values = [1, 2, 5, 10, 20]
rows_mink = []

print(f"{'min_k':>7}  {'MAE':>8}  {'RMSE':>8}  {'Cobertura':>10}")
print('-' * 38)

for mink in mink_values:
    a = KNNWithMeans(k=50, min_k=mink, sim_options={'name':'pearson','user_based':True}, verbose=False)
    a.fit(trainset)
    p = a.test(testset)
    mae  = accuracy.mae(p,  verbose=False)
    rmse = accuracy.rmse(p, verbose=False)
    cov  = sum(1 for x in p if not x.details['was_impossible']) / len(p)
    rows_mink.append({'min_k': mink, 'MAE': mae, 'RMSE': rmse, 'Cobertura': f'{cov:.1%}'})
    print(f'{mink:>7}  {mae:>8.4f}  {rmse:>8.4f}  {cov:>10.1%}')

df_mink = pd.DataFrame(rows_mink).set_index('min_k')
df_mink

  min_k       MAE      RMSE   Cobertura
--------------------------------------
      1    0.6117    0.7994      100.0%
      2    0.6116    0.7993      100.0%
      5    0.6116    0.7992      100.0%
     10    0.6121    0.8000      100.0%
     20    0.6138    0.8027      100.0%


,MAE,RMSE,Cobertura
min_k,,,
1,0.611667,0.799356,100.0%
2,0.611623,0.799274,100.0%
5,0.611569,0.799193,100.0%
10,0.612096,0.800023,100.0%
20,0.613819,0.802675,100.0%


## 6. Punto 3e – McLaughlin Significance Weighting

Surprise implementa significance weighting con el parámetro `shrinkage` en `pearson_baseline`.

La fórmula es:
$$w'_{u,v} = w_{u,v} \times \frac{n_{u,v}}{n_{u,v} + \gamma}$$

donde  $w_{u,v}$ es la similitud original, ​$n_{u,v}$ es número de ítems co-calificados y $\gamma$ es el factor de contracción (shrinkage).

A mayor $\gamma$, más penalizadas las similitudes con pocos co-ratings.

In [21]:
gammas = [0, 25, 50, 100, 200]
rows_sw = []

print(f"{'Gamma':>7}  {'MAE':>8}  {'RMSE':>8}  {'Cobertura':>10}")
print('-' * 38)

for gamma in gammas:
    sim_opts = {
        'name': 'pearson_baseline',
        'user_based': True,
        'shrinkage': gamma       # significance weighting
    }
    algo = KNNWithMeans(k=20, min_k=5, sim_options=sim_opts, verbose=False)
    algo.fit(trainset)
    preds = algo.test(testset)

    mae  = accuracy.mae(preds,  verbose=False)
    rmse = accuracy.rmse(preds, verbose=False)
    cov  = sum(1 for p in preds if not p.details['was_impossible']) / len(preds)

    rows_sw.append({'Gamma': gamma, 'MAE': mae, 'RMSE': rmse, 'Cobertura': f'{cov:.1%}'})
    print(f'{gamma:>7}  {mae:>8.4f}  {rmse:>8.4f}  {cov:>10.1%}')

df_sw = pd.DataFrame(rows_sw).set_index('Gamma')
df_sw

  Gamma       MAE      RMSE   Cobertura
--------------------------------------
      0    0.5945    0.7775      100.0%
     25    0.5927    0.7756      100.0%
     50    0.5917    0.7746      100.0%
    100    0.5906    0.7737      100.0%
    200    0.5896    0.7731      100.0%


,MAE,RMSE,Cobertura
Gamma,,,
0,0.594529,0.777495,100.0%
25,0.592668,0.775576,100.0%
50,0.591665,0.774584,100.0%
100,0.590590,0.773669,100.0%
200,0.589572,0.773054,100.0%


## 7. Resumen final consolidado

In [22]:
print('='*55)
print('3c) Comparación de métricas de similitud (k=10 , min_k=2)')
print('='*55)
print(df_metrics.to_string())

print()
print('='*55)
print('3d-i) Efecto del número de vecinos K (Pearson con min_k=2)')
print('='*55)
print(df_k.to_string())

print()
print('='*55)
print('3d-ii) Efecto de min_k / umbral de vecinos (k=50)')
print('='*55)
print(df_mink.to_string())

print()
print('='*55)
print('3e) McLaughlin Significance Weighting (gamma = shrinkage)')
print('='*55)
print(df_sw.to_string())

3c) Comparación de métricas de similitud (k=10 , min_k=2)
              MAE    RMSE Cobertura
Similitud                          
Jaccard    0.7624  0.9897     87.7%
Coseno     0.6317  0.8152    100.0%
Pearson    0.6268  0.8171    100.0%

3d-i) Efecto del número de vecinos K (Pearson con min_k=2)
     MAE_Coseno  MAE_Pearson  RMSE_Coseno  RMSE_Pearson
K                                                      
5      0.646994     0.648608     0.833755      0.843758
10     0.631653     0.626808     0.815219      0.817098
20     0.623588     0.616020     0.806380      0.804314
50     0.620157     0.611623     0.803626      0.799274
100    0.621422     0.612166     0.806122      0.799856

3d-ii) Efecto de min_k / umbral de vecinos (k=50)
            MAE      RMSE Cobertura
min_k                              
1      0.611667  0.799356    100.0%
2      0.611623  0.799274    100.0%
5      0.611569  0.799193    100.0%
10     0.612096  0.800023    100.0%
20     0.613819  0.802675    100.0%

3e) Mc

Despues de probar los diferentes parametros se obtuvo que :

In [25]:
# En vez de esto:
ratings = pd.read_csv("rating_sample.csv")[['userId','movieId','rating']]


In [5]:
## 8. Guardar el mejor modelo

# Ganador según experimentos:
# pearson_baseline, k=50, min_k=5, gamma=200
# MAE=0.5880, RMSE=0.7718

import pickle
from surprise import Dataset, Reader, KNNWithMeans

BEST_K     = 50
BEST_MINK  = 5
BEST_GAMMA = 200
ratings = pd.read_csv("rating_sample.csv")[['userId','movieId','rating']]
print(f"Entrenando con TODOS los datos: {len(ratings):,} ratings")

reader_full   = Reader(rating_scale=(0.5, 5.0))
data_full     = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader_full)
trainset_full = data_full.build_full_trainset()

best_algo = KNNWithMeans(
    k=BEST_K, min_k=BEST_MINK,
    sim_options={'name': 'pearson_baseline', 'user_based': True, 'shrinkage': BEST_GAMMA},
    verbose=True
)
best_algo.fit(trainset_full)
print("Modelo entrenado.")

# Catálogo de películas
movies_dict = {}
try:
    movies_df = pd.read_csv("movie.csv")
    for _, row in movies_df.iterrows():
        genres = row["genres"].split("|") if isinstance(row.get("genres",""), str) else []
        movies_dict[int(row["movieId"])] = {"title": row["title"], "genres": genres}
    print(f"{len(movies_dict):,} películas cargadas")
except FileNotFoundError:
    print("AVISO: movie.csv no encontrado")

# Historial de ratings por usuario (para el frontend)
user_ratings_index = {
    int(uid): grp[["movieId","rating"]].to_dict("records")
    for uid, grp in ratings.groupby("userId")
}

# Precalcular vecinos de cada usuario con su similitud
# Esto alimenta el modal "¿Por qué?" del frontend
print("Precalculando vecinos...")
user_neighbors = {}
for inner_uid in trainset_full.all_users():
    raw_uid   = trainset_full.to_raw_uid(inner_uid)
    sim_row   = best_algo.sim[inner_uid]
    top_k_idx = np.argsort(sim_row)[::-1][1:BEST_K+1]
    neighbors = []
    for nb_inner in top_k_idx:
        sim_val = float(sim_row[nb_inner])
        if sim_val <= 0:
            continue
        neighbors.append({
            "userId":     trainset_full.to_raw_uid(nb_inner),
            "similarity": round(sim_val, 4),
        })
    user_neighbors[int(raw_uid)] = neighbors
print(f"Vecinos listos para {len(user_neighbors):,} usuarios")

pickle.dump({
    "trainset":       trainset_full,
    "movies_dict":    movies_dict,
    "user_ratings":   user_ratings_index,
    "user_neighbors": user_neighbors,
    "best_model":     best_algo,
    "best_params": {
        "similarity": "pearson_baseline",
        "k":    BEST_K,
        "min_k": BEST_MINK,
        "gamma": BEST_GAMMA,
        "mae":   0.5880,
        "rmse":  0.7718,
    },
}, open("model_cache.pkl", "wb"))

print(f"\n✅  model_cache.pkl listo")
print(f"   {len(user_ratings_index):,} usuarios | pearson_baseline k={BEST_K} gamma={BEST_GAMMA}")
print(f"   Descarga model_cache.pkl y movie.csv y ponlos junto a main.py")

Entrenando con TODOS los datos: 1,836,903 ratings
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Modelo entrenado.
27,278 películas cargadas
Precalculando vecinos...
Vecinos listos para 2,000 usuarios

✅  model_cache.pkl listo
   2,000 usuarios | pearson_baseline k=50 gamma=200
   Descarga model_cache.pkl y movie.csv y ponlos junto a main.py


In [6]:
## Test rápido — probar recomendaciones sin frontend

# Toma un usuario del modelo
test_uid = list(user_ratings_index.keys())[0]
print(f"Usuario de prueba: {test_uid}")
print(f"Ha calificado {len(user_ratings_index[test_uid])} películas")

# Sus películas ya calificadas
rated = {r["movieId"] for r in user_ratings_index[test_uid]}

# Predecir para películas no vistas
preds = []
for inner_iid in trainset_full.all_items():
    raw_iid = trainset_full.to_raw_iid(inner_iid)
    if raw_iid in rated:
        continue
    pred = best_algo.predict(test_uid, raw_iid)
    if not pred.details.get("was_impossible", False):
        preds.append((raw_iid, pred.est))

preds.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 10 recomendaciones para usuario {test_uid}:")
for i, (mid, est) in enumerate(preds[:10]):
    titulo = movies_dict.get(mid, {}).get("title", f"Movie {mid}")
    generos = movies_dict.get(mid, {}).get("genres", [])
    print(f"  {i+1}. {titulo} ({', '.join(generos[:3])}) → {est:.2f}")

Usuario de prueba: 347
Ha calificado 778 películas

Top 10 recomendaciones para usuario 347:
  1. Seven Samurai (Shichinin no samurai) (1954) (Action, Adventure, Drama) → 4.48
  2. Civil War, The (1990) (Documentary, War) → 4.40
  3. Paths of Glory (1957) (Drama, War) → 4.39
  4. Cosmos (1980) (Documentary) → 4.35
  5. To Kill a Mockingbird (1962) (Drama) → 4.33
  6. Hellsing Ultimate OVA Series (2006) (Action, Animation, Horror) → 4.33
  7. Cowboy Bebop (1998) (Action, Adventure, Animation) → 4.32
  8. Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966) (Action, Adventure, Western) → 4.30
  9. Yojimbo (1961) (Action, Adventure) → 4.28
  10. Third Man, The (1949) (Film-Noir, Mystery, Thriller) → 4.26
